# 原始季度实际值与预期口径资格

固定105次财报，不看收益。按公司原文保留GAAP/非GAAP EPS及营收、季度标签和打印精度。禁止把预测/全年数据当季度实际，禁止按接近预期程度选口径。先解析和对齐，未认证前不构造交易信号。

In [1]:
from pathlib import Path
import os,json,hashlib
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import qlib,pandas as pd,yaml
from qlib.data import D
from qlib.workflow import R
from src.data.issuer_results_intake import collect_issuer_results
from src.data.nas_market_data import archive_nas_recorder
from src.workflow.minute_action_value import _source
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/issuer_actuals_qualification_20261013';plan=json.loads((card/'plan.json').read_text())
qlib.init(provider_uri=str(root/plan['price_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(freq='day'))
objects,consensus_audit=_source(root,plan['consensus_source'],['analysis/events'])
raw=root/plan['raw_cache'];manifest=collect_issuer_results(raw,plan)
rows=[]
for e in manifest['entries']:
    if e['event_role']!='quarterly':continue
    assert pd.Timestamp(e['report_date']) in calendar
    for basis,values in e['parsed']['basis'].items():rows.append({'symbol':e['symbol'],'report_date':e['report_date'],'basis':basis,'source_url':e['url'],'source_sha256':e['sha256'],**values})
actuals=pd.DataFrame(rows)
paired=actuals.merge(objects['analysis/events'][['symbol','report_date','quarter','consensus_eps','smart_eps','consensus_revenue_million_usd']],on=['symbol','report_date'],how='left',validate='many_to_one')
audit={'expected_quarterly_releases':len(plan['events']),'parsed_quarterly_releases':len(actuals[['symbol','report_date']].drop_duplicates()),'basis_rows':len(actuals),'per_symbol_basis_rows':actuals.groupby(['symbol','basis']).size().rename('rows').reset_index().to_dict('records'),'failures':manifest['failures'],'preliminary_releases':[{k:e[k] for k in ['symbol','report_date','url']} for e in manifest['entries'] if e['event_role']=='preliminary'],'forecast_basis_match_certified':False,'historical_original_bytes_certified':False,'all_unscheduled_disclosures_complete':False,'model_fits':0,'backtests':0}
audit['missing_revenue_basis_rows']=int(actuals.revenue_million_usd.isna().sum())
audit['missing_eps_basis_rows']=int(actuals.eps_usd.isna().sum())
audit['alternative_eps_basis_rows']=int(actuals.eps_alternatives.map(bool).sum())
nvda_reference=pd.DataFrame([e for e in plan['events'] if e['symbol']=='NVDA'])
check=actuals.loc[(actuals.symbol=='NVDA') & (actuals.basis=='gaap')].merge(nvda_reference,on=['symbol','report_date'],validate='one_to_one')
check['difference_vs_frozen_issuer_fact']=check.revenue_million_usd-check.frozen_original_revenue_million
check['within_printed_precision']=check.difference_vs_frozen_issuer_fact.abs().le(check.revenue_rounding_half_million+1e-6)
audit['nvda_revenue_crosscheck_rows']=len(check)
audit['nvda_revenue_crosscheck_pass']=bool(check.within_printed_precision.all())
audit['issuer_calendar_matched_releases']=audit['parsed_quarterly_releases']
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],recorder_name='original_quarterly_actuals_basis_audit',uri=str(tracking)):
    rec=R.get_recorder();rec.log_params(plan_sha256=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),model_fits=0,backtests=0)
    rec.save_objects(artifact_path='sources',manifest=manifest,html={p.name:p.read_bytes() for p in raw.glob('*.html')},consensus_audit=consensus_audit,parser=(root/'src/data/issuer_result_tables.py').read_text(),collector=(root/'src/data/issuer_results_intake.py').read_text())
    rec.save_objects(artifact_path='analysis',actuals=actuals,paired=paired,audit=audit,calendar=calendar,nvda_crosscheck=check)
    rec.log_metrics(registered_artifact_count=10,parsed_quarterly_releases=audit['parsed_quarterly_releases'])
summary={'experiment':plan['experiment'],'decision':'actuals_parsed_needs_expectation_basis_and_novelty_qualification' if audit['parsed_quarterly_releases']==len(plan['events']) else 'needs_source_repair','source_audit':audit,'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/str(rec.experiment_id)/rec.id),'provider_uri':str(root/plan['price_provider']),'as_of':plan['as_of'],'region':'us','qlib_docs':'https://qlib.readthedocs.io/en/latest/component/workflow.html','objective_achieved':False,'goal_remains_active':True,'promotion_eligible':False,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['sources/*','analysis/*']}}
artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]},indent=2)+'\n')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id))
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':plan['experiment'],'provider_uri':summary['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']}));paired.to_csv(card/'basis_comparison.csv',index=False)
audit


{'expected_quarterly_releases': 105,
 'parsed_quarterly_releases': 105,
 'basis_rows': 210,
 'per_symbol_basis_rows': [{'symbol': 'AMD', 'basis': 'gaap', 'rows': 35},
  {'symbol': 'AMD', 'basis': 'non_gaap', 'rows': 35},
  {'symbol': 'INTC', 'basis': 'gaap', 'rows': 35},
  {'symbol': 'INTC', 'basis': 'non_gaap', 'rows': 35},
  {'symbol': 'NVDA', 'basis': 'gaap', 'rows': 35},
  {'symbol': 'NVDA', 'basis': 'non_gaap', 'rows': 35}],
 'failures': [],
 'preliminary_releases': [{'symbol': 'AMD',
   'report_date': '2022-10-06',
   'url': 'https://ir.amd.com/news-events/press-releases/detail/1093/amd-announces-preliminary-third-quarter-2022-financial-results'},
  {'symbol': 'NVDA',
   'report_date': '2022-08-08',
   'url': 'https://nvidianews.nvidia.com/news/nvidia-announces-preliminary-financial-resultsfor-second-quarter-fiscal-2023'},
  {'symbol': 'NVDA',
   'report_date': '2019-01-28',
   'url': 'https://nvidianews.nvidia.com/news/nvidia-updates-financial-guidance-for-fourth-quarter-of-fisc

# 公司原始实际值与预期口径核对

105次季度财报全部和公司公告对应，取得210条GAAP/非GAAP记录，每家公司每种口径各35条。两种EPS均有值；14条Intel非GAAP营收在原表未单列，保持未知，不根据空白推断等于GAAP。2条NVIDIA非GAAP记录另列剔除H20影响的EPS，已单独保留，不覆盖主EPS。

NVIDIA的35条GAAP营收与既有冻结公司披露事实全部在原文精度内一致。AMD/INTC的70个财报日期也和公司发布目录对应，其中Intel2020-10-22用遗漏于分类目录的官方公告补齐。AMD2024-10-29现行新闻页的GAAP表格结构损坏，改用公司托管的原始EX99.1文件。原始公告、目录、数值、表列位置及精度均在NAS Qlib Recorder。

解析修复涵盖旧季度写法、金额后缀、负数括号、脚注、thead标签、纵向双口径表格及标签优先级。AMD2017Q4曾受说明段落的“非GAAP”字样干扰；已修复为明确表头优先，其后用最近的完整说明段落，再用分散标签。各修复前Recorder保留在qualification_attempts.json。没有改金额来迎合预期，没有按哪个口径回测更好做选择。

原始精度保留：按十亿美元只印一位小数的营收保留相应舍入半宽，未伪造百万美元精度。空白当前值不沿用上期；全年表与下一季度展望不作季度实际值。Qlib原生日历用于事件日期检验。

保留了NVDA2019-01-28、2022-08-08及AMD2022-10-06三次明确提前披露。它们不等于完整非定期消息历史。预期供应商的标准化口径、历史首次公开时间和当前网页与当时字节一致性尚未全认证，forecast_basis_match_certified、historical_original_bytes_certified及promotion_eligible仍为false。

本轮0拟合、0回测，没有新增收益。决定actuals_parsed_needs_expectation_basis_and_novelty_qualification。下一步对齐预期的会计口径和已有预警，构造仅使用当时已知信息的特征，再冻结有限对照实验。生产SIM未改，稳定净SOXX超额目标未完成，保持active。
